In [10]:
!pip install gensim scikit-learn sentence-transformers

# IMPORTS

import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import re

# LOAD DATA

def load_data(path):
    try:
        df = pd.read_csv(path)
        assert 'review_text' in df.columns
        return df
    except Exception as e:
        print("Error loading data:", e)
        return None

df = load_data('/content/shopsense_reviews.csv')

# TEXT PREPROCESSING

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.split()

df['tokens'] = df['review_text'].astype(str).apply(preprocess)

# TRAIN WORD2VEC

def train_word2vec(sentences, window_size=5):
    model = Word2Vec(
        sentences=sentences,
        vector_size=100,
        window=window_size,
        min_count=2,
        workers=4
    )
    return model

w2v_model = train_word2vec(df['tokens'])

# Q1 (a): COSINE SIMILARITY

def compute_similarity(model, w1, w2):
    try:
        return model.wv.similarity(w1, w2)
    except:
        return None

sim_affordable = compute_similarity(w2v_model, 'cheap', 'affordable')
sim_flimsy = compute_similarity(w2v_model, 'cheap', 'flimsy')

print("cheap vs affordable:", sim_affordable)
print("cheap vs flimsy:", sim_flimsy)

# Q1 (b): DISAMBIGUATION

def get_avg_vector(model, words):
    vectors = [model.wv[w] for w in words if w in model.wv]
    return np.mean(vectors, axis=0)

affordable_anchor = get_avg_vector(w2v_model, ['affordable','budget','value'])
low_quality_anchor = get_avg_vector(w2v_model, ['flimsy','poor','bad'])

def disambiguate(sentence, model):
    tokens = preprocess(sentence)
    context_vec = get_avg_vector(model, tokens)

    sim_aff = cosine_similarity([context_vec], [affordable_anchor])[0][0]
    sim_low = cosine_similarity([context_vec], [low_quality_anchor])[0][0]

    return "affordable" if sim_aff > sim_low else "low-quality"

# Example
print(disambiguate("This phone is cheap and worth the price", w2v_model))
print(disambiguate("This feels cheap and breaks easily", w2v_model))

# Q1 (c): WINDOW COMPARISON

model_w2 = train_word2vec(df['tokens'], window_size=2)
model_w10 = train_word2vec(df['tokens'], window_size=10)

print("Window=2 similarity:", compute_similarity(model_w2, 'good', 'product'))
print("Window=10 similarity:", compute_similarity(model_w10, 'good', 'product'))

# Q2: SIMILARITY METHODS

review_a = "incredible camera but terrible battery life"
review_b = "Battery drains fast although photos are stunning"

# ---------- BOW ----------
def bow_similarity(a, b):
    vec = CountVectorizer()
    X = vec.fit_transform([a, b])
    return cosine_similarity(X)[0][1]

# ---------- TF-IDF ----------
def tfidf_similarity(a, b):
    vec = TfidfVectorizer()
    X = vec.fit_transform([a, b])
    return cosine_similarity(X)[0][1]

# ---------- Word2Vec ----------
def sentence_vector(model, sentence):
    tokens = preprocess(sentence)
    return get_avg_vector(model, tokens)

def w2v_similarity(model, a, b):
    v1 = sentence_vector(model, a)
    v2 = sentence_vector(model, b)
    return cosine_similarity([v1], [v2])[0][0]

# ---------- Sentence-BERT ----------
def sbert_similarity(a, b):
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode([a, b])
    return cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]

# RESULTS
print("BOW:", bow_similarity(review_a, review_b))
print("TF-IDF:", tfidf_similarity(review_a, review_b))
print("Word2Vec:", w2v_similarity(w2v_model, review_a, review_b))
print("Sentence-BERT:", sbert_similarity(review_a, review_b))

cheap vs affordable: None
cheap vs flimsy: None
low-quality
low-quality
Window=2 similarity: 0.27255455
Window=10 similarity: -0.038819984
BOW: 0.1543033499620919
TF-IDF: 0.0845798608014702
Word2Vec: 0.38753745


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentence-BERT: 0.6300819
